In [1]:
import re
import emoji
import contractions
from textblob import TextBlob
import spacy
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI


c:\Users\ashmi\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\ashmi\AppData\Local\Temp\ipykernel_1248\1664497480.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


### 1. Load the document

In [14]:
data=open('data.txt',encoding='latin').read()

### 2. Text Normalization

*Converting to lowercase*

In [15]:
data=data.lower()

*Removing extra space*

In [16]:
data=re.sub(r'\s{2,}',' ',data)


*Handling emoji*

In [17]:
'To remove emoji '

data=emoji.replace_emoji(data)

'To demojize '

'data=emoji.demojize(data)'

'data=emoji.demojize(data)'

*Removing special characters and punctuations*

In [18]:
data=re.sub(r'[^0-9a-z\s]','',data)

*Contractions*

In [19]:
data=contractions.fix(data)

*Correcting the words*

In [20]:
values=TextBlob(data).correct().raw_sentences
data=' '.join(values)

### 3. Tokenization

*Lemmatization + stopword removal*

In [23]:
nlp=spacy.load('en_core_web_sm')
tokens=nlp(data)

updated_tokens=[token.lemma_ for token in tokens if not token.is_stop]
data=' '.join(updated_tokens).strip()



### 4. Chunking


In [29]:
splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks=splitter.create_documents([data])

### 5. Embeddings

In [30]:
embedding_model=HuggingFaceEmbeddings(
    model='sentence-transformers/all-MiniLM-L6-V2'
)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3139.77it/s]


### 6. Vector DataBase

In [31]:
vector_db=FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

### 7. Retrival and Generation

In [34]:
def search_retrive(query,k=4):
    r_chunks=vector_db.similarity_search(query,k=k)
    r_chunks={doc.page_content for doc in r_chunks}
    r_text=' '.join(r_chunks)
    return r_text
def generation(search_retrive,query):
    import os
    import requests
    prompt = f'''
            You are a helpful assistant.
            Answer the user's question using ONLY the provided context.
            Context:
            {search_retrive}
            Question:
            {query}
            Instructions:
            - Answer the question directly and in detail.
            - Explain the main concept clearly and completely.
            - Include important details, related concepts, examples, or steps only when they are present in the context.
            - Organize the answer using headings or bullet points when appropriate.
            - Do not unnecessarily repeat information.
            - Do not add information that is not present in the context.
            - Do not make assumptions or invent facts.
            - Make the answer easy to understand, even for a beginner.
            - If the context does not contain enough information to answer the question, clearly say:
            "The provided context does not contain enough information to answer this question."  
            '''
    llm_model=ChatGoogleGenerativeAI(
        model="gemini-3.5-flash",
        api_key=os.environ['Google Gemini API key']
        )
    response=llm_model.invoke(prompt).content
    return response
user_prompt='What is Machine Learning?'
user_prompt=re.sub(r'[^0-9a-zA-Z\s]','',user_prompt)
retrival_reponse=search_retrive(user_prompt)
genration_response=generation(retrival_reponse,user_prompt)
print(genration_response[0]['text'])

Based on the provided context, here is a detailed explanation of what Machine Learning is:

### **Definition and Core Concept**
Machine learning is a branch of artificial intelligence focused on enabling computers to learn patterns from data. 

Instead of relying on explicitly programmed rules, machine learning algorithms learn from examples and use those examples to make predictions. 

---

### **How Machine Learning Works**
The basic idea of machine learning is simple:
1. **Provide Data:** You provide data to an algorithm. Machine learning systems require large amounts of data to work.
2. **Learn Patterns:** The algorithm is allowed to learn patterns from this data.
3. **Train and Predict:** These learned patterns are used to train a model, which can then make predictions on new, previously unseen data.

---

### **Major Categories of Machine Learning**
Machine learning is divided into major categories. The choice of method depends on the objective of the project and the type of data